In [1]:
import os

import torch
import sqlite3
import pandas as pd

from datetime import datetime

from pathlib import Path
import sys 

ROOT = Path.cwd().parents[1]
sys.path.append(str(ROOT))

from src.config.paths import EMBEDS_DIR, EXPERIMENTS, RESULTS, DB_PATH

time = datetime.now().strftime("%Y%m%d_%H%M%S")
EMBED_PATH = EMBEDS_DIR / "dino/20260801_221150"
EMBED_NAME = EMBED_PATH.stem

EXPERIMENTS_DIR = EXPERIMENTS / EMBED_NAME / f"ellipsoid_bootstrap/{time}"
RESULTS_DIR = RESULTS / EMBED_NAME / f"ellipsoid_bootstrap/{time}"

In [2]:
os.makedirs(EXPERIMENTS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

In [3]:
cls_tokens = torch.load(EMBED_PATH/"cls.pt", weights_only=False)

In [4]:
conn = sqlite3.connect(DB_PATH)

meta = pd.read_sql_query("SELECT * FROM meta", conn)
categories = pd.read_sql_query("SELECT DISTINCT category FROM meta", conn)["category"].to_list()

conn.close()

In [10]:
%load_ext autoreload
%autoreload 2

from src.algorithims.ellipsoid import EllipsoidCover, EllipsoidEvaluator, EllipsoidFitter, CandidateCleaner
from src.algorithims.ellipsoid.bootstrap import BootstrapRunner
from src.types import ExperimentConfig, AlgorithmResults

from src.stats.mahlanobis_detector import MahalanobisDetector

metadata = ExperimentConfig(
    K_frac=0.05,
    start_growth=1.2,
    min_growth=1,
    reg=1e-4,
    growth_type="variance_scaled",
    cleaner="shared_axis"
)

fitter = EllipsoidFitter(support_points=5, reg=1e-4)
cleaner = CandidateCleaner(fitter=fitter, min_points=1)

cover = EllipsoidCover(fitter=fitter, cleaner=cleaner)
evaluator = EllipsoidEvaluator()

mal_detector = MahalanobisDetector(reg=1e-6)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
test_df : list[pd.DataFrame] = []
train_df: list[pd.DataFrame] = []

for category in categories:
    print("Running", category)
    outputs_dir = EXPERIMENTS_DIR / category 
    os.makedirs(outputs_dir, exist_ok=True)

    train_mask = meta["split"] == "train"
    train_meta = meta[train_mask]
    test_meta = meta[~train_mask]

    train_cat_mask = train_meta["category"] == category

    train_emb = cls_tokens[train_mask]
    cat_emb = train_emb[train_cat_mask]

    good_test_cat_mask = (test_meta["category"] == category) & (test_meta["type"] == "good")
    defect_test_cat_mask = (test_meta["category"] == category) & (test_meta["type"] != "good")

    test_emb = cls_tokens[~train_mask]
    defect_test_emb = test_emb[defect_test_cat_mask]
    good_test_emb = test_emb[good_test_cat_mask]

    runner = BootstrapRunner(
        cover=cover,
        evaluator=evaluator,
        mal_detector=mal_detector,
        n_test_bootstraps=1000,
        n_train_bootstraps=100,
        seed=42,
    )

    test_bootstraps, train_bootstraps = runner.run(
        train_emb=cat_emb, 
        good_test_emb=good_test_emb, 
        defect_test_emb=defect_test_emb
        )

    train_bootstraps.insert(0, "category", category)
    test_bootstraps.insert(0, "category", category)

    train_bootstraps.to_csv(RESULTS_DIR / f"{category}_train_bootstraps.csv", index=False)
    test_bootstraps.to_csv(RESULTS_DIR / f"{category}_test_bootstraps.csv", index=False)

    test_df.append(test_bootstraps)
    train_df.append(train_bootstraps)

Running bottle


Test Bootstraps:   0%|          | 0/1000 [00:00<?, ?it/s]

Train Bootstraps:   0%|          | 0/100 [00:00<?, ?it/s]

Running cable


Test Bootstraps:   0%|          | 0/1000 [00:00<?, ?it/s]

...

Test Bootstraps:   0%|          | 0/1000 [00:00<?, ?it/s]

Train Bootstraps:   0%|          | 0/100 [00:00<?, ?it/s]

Running zipper


Test Bootstraps:   0%|          | 0/1000 [00:00<?, ?it/s]

Train Bootstraps:   0%|          | 0/100 [00:00<?, ?it/s]

In [14]:
train_summaries: list[pd.DataFrame] = []
test_summaries: list[pd.DataFrame] = []

for category, train_bootstraps, test_bootstraps in zip(
    categories,
    train_df,
    test_df,
):
    train_summary = runner.summarise_bootstrap(train_bootstraps)
    test_summary = runner.summarise_bootstrap(test_bootstraps)

    # In case summarise_bootstrap returns a Series
    if isinstance(train_summary, pd.Series):
        train_summary = train_summary.to_frame().T

    if isinstance(test_summary, pd.Series):
        test_summary = test_summary.to_frame().T

    train_summary.insert(0, "category", category)
    test_summary.insert(0, "category", category)

    train_summaries.append(train_summary)
    test_summaries.append(test_summary)

train_summary_df = pd.concat(
    train_summaries,
    ignore_index=True,
)

test_summary_df = pd.concat(
    test_summaries,
    ignore_index=True,
)

In [15]:
train_summary_df.to_csv(
    RESULTS_DIR / "train_bootstrap_summary.csv",
    index=False,
)

train_summary_df

,category,alg_score_mean,alg_score_std,alg_score_ci_lower,alg_score_ci_upper,mal_score_mean,mal_score_std,mal_score_ci_lower,mal_score_ci_upper,delta_mean,...,pc1_ratio_mean_mean,pc1_ratio_mean_std,pc1_ratio_mean_ci_lower,pc1_ratio_mean_ci_upper,rank_mean_mean,rank_mean_std,rank_mean_ci_lower,rank_mean_ci_upper,mal_score,median_n_points
0,bottle,0.958214,0.027091,0.889643,0.988095,0.999944,0.000204,0.999206,1.000000,-0.041730,...,0.585658,0.019812,0.552294,0.623099,4.425742,0.326595,3.863796,5.289939,NaN,NaN
1,cable,0.765759,0.031873,0.675394,0.817012,0.918593,0.005696,0.908222,0.929367,-0.152834,...,0.548896,0.021841,0.510338,0.590736,4.487582,0.359493,3.844643,5.412791,NaN,NaN
2,capsule,0.796709,0.044698,0.674392,0.856402,0.950758,0.010073,0.931183,0.966095,-0.154049,...,0.586375,0.020797,0.542290,0.629833,4.450973,0.376424,3.724405,5.166279,NaN,NaN
3,carpet,0.993355,0.007479,0.971087,1.000000,0.993371,0.001185,0.990961,0.995395,-0.000016,...,0.539126,0.018876,0.500571,0.575779,5.278438,0.358240,4.454965,5.924457,NaN,NaN
4,grid,0.969699,0.028734,0.906266,0.997139,0.984348,0.001607,0.982456,0.988304,-0.014649,...,0.586659,0.017822,0.555527,0.620742,5.106716,0.352663,4.378804,5.703853,NaN,NaN
5,hazelnut,0.968811,0.010998,0.937804,0.982330,0.990229,0.002263,0.985714,0.993929,-0.021418,...,0.494501,0.015881,0.470983,0.530161,6.178793,0.335528,5.602830,6.802358,NaN,NaN
6,leather,0.952232,0.041369,0.830562,0.994599,NaN,NaN,NaN,NaN,-0.047768,...,0.568429,0.021133,0.530390,0.612263,4.798830,0.318959,4.166288,5.330271,1.0,NaN
7,metal_nut,0.850973,0.043967,0.740787,0.910838,0.992566,0.004222,0.983871,0.998534,-0.141593,...,0.589526,0.022002,0.546752,0.628184,4.285824,0.311529,3.713095,4.943605,NaN,NaN
8,pill,0.801871,0.045982,0.689546,0.865992,0.961776,0.005706,0.950675,0.972736,-0.159905,...,0.534646,0.017657,0.505527,0.570954,5.061469,0.342777,4.498913,5.794022,NaN,NaN
9,screw,0.847295,0.032857,0.783019,0.894866,0.921033,0.010189,0.900477,0.936063,-0.073738,...,0.558919,0.018194,0.522656,0.591323,5.556986,0.368258,5.040816,6.419625,NaN,NaN


In [13]:
test_summary_df.to_csv(
    RESULTS_DIR / "test_bootstrap_summary.csv",
    index=False,
)

test_summary_df

,alg_score_mean,alg_score_std,alg_score_ci_lower,alg_score_ci_upper,mal_score_mean,mal_score_std,mal_score_ci_lower,mal_score_ci_upper,delta_mean,delta_std,delta_ci_lower,delta_ci_upper,mal_score
0,0.980972,0.012400,0.950794,0.998413,1.000000,1.405036e-17,1.000000,1.000000,-0.019028,0.012400,-4.920635e-02,-0.001587,NaN
1,0.805113,0.033952,0.738943,0.869776,0.929725,1.855959e-02,0.888676,0.962153,-0.124612,0.024990,-1.778626e-01,-0.077956,NaN
2,0.816296,0.040708,0.729148,0.889120,0.962300,1.539073e-02,0.929388,0.987236,-0.146003,0.040354,-2.297667e-01,-0.072188,NaN
3,0.999616,0.000658,0.997592,1.000000,0.993982,6.245846e-03,0.978311,1.000000,0.005634,0.005882,-1.110223e-16,0.020465,NaN
4,0.981716,0.017558,0.936508,1.000000,0.984314,1.582808e-02,0.946533,1.000000,-0.002598,0.003129,-1.004595e-02,0.001671,NaN
5,0.972795,0.012366,0.944643,0.992161,0.986958,7.882660e-03,0.967857,0.998929,-0.014164,0.009766,-3.500000e-02,0.001804,NaN
6,0.995621,0.003657,0.986413,1.000000,NaN,NaN,NaN,NaN,-0.004379,0.003657,-1.358696e-02,0.000000,1.0
7,0.889827,0.031486,0.822568,0.944282,0.999043,1.369658e-03,0.995112,1.000000,-0.109216,0.031552,-1.759653e-01,-0.053751,NaN
8,0.829934,0.039869,0.744108,0.902912,0.959211,1.422722e-02,0.929330,0.983640,-0.129277,0.033986,-1.942239e-01,-0.069013,NaN
9,0.860126,0.030775,0.794420,0.911667,0.925713,2.570794e-02,0.871480,0.971521,-0.065588,0.033575,-1.367391e-01,-0.000384,NaN
